## Audio books analysis
This model will analyse the data of past clients to train and then be able to forecast whether a customer is likely to buy again or not.

In [12]:
# import libraries
import pandas as pd  # to read the CSV
import numpy as np
import tensorflow as tf


In [29]:
#Load data from csv
abooks_df = pd.read_csv("../../../../statistics/python/audiobooks/Audiobooks_data.csv",header=None)
abooks_df.head()
#ID,Book length(mins)_overal,Book length (mins)_avg,Price_overall,Price_avg,Review,Review 10/10,Minutes listened,Completion,Support Requests,Last visited minus Purchase date,Targets
#0       1                               2                3           4        5            6           7              8              9                10                         11

abooks_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14084 entries, 0 to 14083
Data columns (total 12 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       14084 non-null  int64  
 1   1       14084 non-null  float64
 2   2       14084 non-null  int64  
 3   3       14084 non-null  float64
 4   4       14084 non-null  float64
 5   5       14084 non-null  int64  
 6   6       14084 non-null  float64
 7   7       14084 non-null  float64
 8   8       14084 non-null  float64
 9   9       14084 non-null  int64  
 10  10      14084 non-null  int64  
 11  11      14084 non-null  int64  
dtypes: float64(6), int64(6)
memory usage: 1.3 MB


In [59]:
# Shuffle and separate targets from data
# Shuffle -> no shuffle for df

targets = abooks_df.iloc[:,11]
targets.info()

<class 'pandas.core.series.Series'>
RangeIndex: 14084 entries, 0 to 14083
Series name: 11
Non-Null Count  Dtype
--------------  -----
14084 non-null  int64
dtypes: int64(1)
memory usage: 110.2 KB


In [63]:
abooks_data = abooks_df.iloc[:,0:11]
abooks_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14084 entries, 0 to 14083
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       14084 non-null  int64  
 1   1       14084 non-null  float64
 2   2       14084 non-null  int64  
 3   3       14084 non-null  float64
 4   4       14084 non-null  float64
 5   5       14084 non-null  int64  
 6   6       14084 non-null  float64
 7   7       14084 non-null  float64
 8   8       14084 non-null  float64
 9   9       14084 non-null  int64  
 10  10      14084 non-null  int64  
dtypes: float64(6), int64(5)
memory usage: 1.2 MB


In [64]:
#Normalize columns in the dataframe
def scale_float(data_in):
    data_float = data_in.astype(np.float64)
    return((data_float-data_float.mean())/data_float.std())

abooks_df_normal = abooks_data.apply(scale_float, 0, True, engine='numba', engine_kwargs={'parallel': True})
abooks_df_normal

,0,1,2,3,4,5,6,7,8,9,10
0,-1.628081,0.056944,-0.089504,2.560319,2.191789,2.284918,1.694489,3.583548,3.810353,10.441350,0.340855
1,-1.612707,1.127687,0.735156,-0.359686,-0.398171,-0.437653,0.000319,-0.520980,-0.511732,-0.148730,-0.702175
2,-1.518191,1.127687,0.735156,-0.359686,-0.398171,-0.437653,0.000319,-0.520980,-0.511732,-0.148730,3.696693
3,-1.433271,0.056944,-0.089504,-0.231936,-0.284861,-0.437653,0.000319,1.220335,1.321880,1.969286,0.760335
4,-1.385806,1.127687,0.735156,-0.359686,-0.398171,-0.437653,0.000319,0.391137,0.768886,-0.148730,3.390586
...,...,...,...,...,...,...,...,...,...,...,...
14079,1.181195,0.056944,-0.089504,-0.359686,-0.398171,2.284918,0.140205,2.008072,2.151371,-0.148730,-0.656826
14080,1.227731,-1.013799,-0.914164,-0.112297,-0.178744,2.284918,-4.522648,0.681356,0.332311,-0.148730,-0.373394
14081,1.481872,1.127687,0.735156,-0.195436,-0.252486,-0.437653,0.000319,-0.520980,-0.511732,-0.148730,-0.702175
14082,1.657078,0.056944,-0.089504,-0.359686,-0.398171,2.284918,-1.414080,1.054495,1.147250,-0.148730,0.318181


In [65]:
requests = abooks_df.iloc[:,9]
requests.mean()

np.float64(0.0702215279750071)

In [66]:
requests.max()

np.int64(30)

In [67]:
abooks_df_normal.loc[abooks_df[9]==requests.max(),:]

,0,1,2,3,4,5,6,7,8,9,10
8309,0.583453,1.127687,0.735156,-0.359686,-0.398171,2.284918,1.694489,3.210409,4.727159,63.391751,-0.237347


In [68]:
abooks_df.iloc[8309,:]

0     22427.00
1      2160.00
2      2160.00
3         5.33
4         5.33
5         1.00
6        10.00
7         0.90
8      1944.00
9        30.00
10       41.00
11        0.00
Name: 8309, dtype: float64

In [69]:
abooks_df_normal.iloc[:,4].std()

np.float64(1.0000355031687493)

In [70]:
abooks_np = np.array(abooks_df_normal)
abooks_np

array([[-1.62808146,  0.05694432, -0.08950406, ...,  3.81035339,
        10.44134984,  0.34085525],
       [-1.61270711,  1.12768719,  0.7351559 , ..., -0.51173244,
        -0.14873032, -0.70217541],
       [-1.51819094,  1.12768719,  0.7351559 , ..., -0.51173244,
        -0.14873032,  3.69669302],
       ...,
       [ 1.48187206,  1.12768719,  0.7351559 , ..., -0.51173244,
        -0.14873032, -0.70217541],
       [ 1.6570778 ,  0.05694432, -0.08950406, ...,  1.14725   ,
        -0.14873032,  0.31818067],
       [-1.70474687,  0.1640186 ,  2.5494078 , ..., -0.51173244,
        -0.14873032, -0.70217541]])

In [71]:
# 10% of the data to be used as validation and 10% as validation
num_validation_samples = tf.cast(abooks_np.shape[0]*0.1, tf.int64)

In [74]:
test_data = abooks_np.iloc[0:num_validation_samples,:]
test_data.shape()

AttributeError: 'numpy.ndarray' object has no attribute 'iloc'

### Machine learing part
Preprocesing is done. It can be reused for other problems.
From here on, we will start from the .npz files.